# HeteroSense-FL — Quickstart Notebook

This notebook walks through the four key steps of using HeteroSense-FL:
1. Build a modality-heterogeneous FL dataset
2. Inspect ModalityBundle observations
3. Iterate with TemporalWindowSampler
4. Run automated observation validation (V1-V4)

**Install:** `pip install heterosense-fl`


## 1. Build a heterogeneous dataset

Three clients with round-robin modality assignment:  
`client 0`: LiDAR + bed  |  `client 1`: LiDAR only  |  `client 2`: bed only


In [ ]:
import sys; sys.path.insert(0, '..')
from heterosense import ClientFactory, ConfigurationManager as CM, DatasetBuilder

clients = ClientFactory.make(3, strategy='round_robin', seed=42)
cfg     = CM.from_clients(clients, n_steps=1000)
sc      = cfg.to_sim_config()
data    = DatasetBuilder(sc).build()

for cid, bundles in data.items():
    cc     = next(c for c in sc.clients if c.client_id == cid)
    has_l  = any(b.lidar    is not None for b in bundles)
    has_p  = any(b.pressure is not None for b in bundles)
    print(f'client {cid}: modalities={list(cc.channel_availability)}'
          f'  lidar={has_l}  bed={has_p}  n_steps={len(bundles)}')

## 2. Inspect a ModalityBundle

Each time step yields a `ModalityBundle` containing the sensor arrays and ground-truth label.


In [ ]:
bundle = data['0'][50]   # step 50, client 0 (both modalities)
print('semantic_state :', bundle.semantic_state)
print('posture_state  :', bundle.posture_state)
print('lidar shape    :', bundle.lidar.shape   if bundle.lidar    is not None else None)
print('pressure shape :', bundle.pressure.shape if bundle.pressure is not None else None)

## 3. Temporal window sampling

TemporalWindowSampler slides a window of length `k` over the bundle sequence.  
The centre bundle at `center_idx()` provides the ground-truth label.

Replace the built-in scalar helpers with your own temporal encoder (LSTM, Transformer, etc.).


In [ ]:
from heterosense import TemporalWindowSampler
import numpy as np

sampler = TemporalWindowSampler(data['0'], window=3)
print(f'Total windows: {sum(1 for _ in sampler)}')
print(f'center_idx   : {sampler.center_idx()}')

# Inspect first three windows
sampler = TemporalWindowSampler(data['0'], window=3)
for i, window in enumerate(sampler):
    if i >= 3: break
    z   = TemporalWindowSampler.lidar_z_series(window)   # shape (3,)
    p   = TemporalWindowSampler.pressure_series(window)  # shape (3,)
    lbl = TemporalWindowSampler.center_label(window, sampler.center_idx())
    print(f'  window {i}: label={lbl:<12}  z_mean={z.mean():.3f}  p_mean={p.mean():.3f}')

## 4. Observation validation (V1–V4)

Four automated checks verify that the generated observations are physically plausible:

| Check | Description |
|-------|-------------|
| V1 | LiDAR geometry validity (UPRIGHT z > LYING z) |
| V2 | Fall pattern morphology (ABNORMAL phase 1 floor-point ratio) |
| V3 | Pressure separability (ON_BED mean > OFF_BED mean) |
| V4 | Per-client modality integrity (absent sensors return None) |


In [ ]:
from heterosense import run_validation

client_modalities = {c.client_id: list(c.channel_availability)
                     for c in sc.clients}
results = run_validation(data, client_modalities)

for r in results:
    icon = 'OK  ' if r.passed else 'FAIL'
    print(f'  [{icon}] {r.name}')
    print(f'         {r.reason}')

## 5. Next steps

- **Reproduce Table 3 benchmarks:** `heterosense-benchmark`  
- **Plug in your algorithm:** see `examples/temporal_plugin_example.py`  
- **Full API docs:** https://heterosense-fl.readthedocs.io
